In [1]:
# =========================================================
# Setup (FAST VERSION – Kaggle + Colab)
# =========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision.models import resnet18
from torch.utils.data import DataLoader
import numpy as np
import random
import pandas as pd
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

# IMPORTANT: enable performance mode
torch.backends.cudnn.benchmark = True

Device: cuda


In [2]:
# =========================================================
# Transforms (STL10 → 96x96)
# =========================================================

class TwoCropsTransform:
    def __init__(self, base_transform):
        self.base_transform = base_transform

    def __call__(self, x):
        return self.base_transform(x), self.base_transform(x)


ssl_base_transform = T.Compose([
    T.RandomResizedCrop(96, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.4,0.4,0.4,0.1),
    T.RandomGrayscale(p=0.2),
    T.ToTensor()
])

ssl_transform = TwoCropsTransform(ssl_base_transform)
supervised_transform = T.ToTensor()

In [3]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = resnet18(weights=None)
        backbone.fc = nn.Identity()
        self.backbone = backbone

    def forward(self, x):
        return self.backbone(x)

In [4]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=2048, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, x):
        return self.net(x)


class BYOL(nn.Module):
    def __init__(self):
        super().__init__()
        self.online_encoder = Encoder()
        self.online_projector = MLP(512)
        self.online_predictor = MLP(256, 512, 256)

        self.target_encoder = Encoder()
        self.target_projector = MLP(512)

        for p in self.target_encoder.parameters():
            p.requires_grad = False
        for p in self.target_projector.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update_target(self, tau=0.996):
        for online, target in zip(self.online_encoder.parameters(),
                                  self.target_encoder.parameters()):
            target.data = tau * target.data + (1 - tau) * online.data

    def forward(self, x1, x2):
        o1 = self.online_predictor(
            self.online_projector(self.online_encoder(x1))
        )

        with torch.no_grad():
            t2 = self.target_projector(
                self.target_encoder(x2)
            )

        loss = 2 - 2 * F.cosine_similarity(o1, t2.detach(), dim=-1)
        return loss.mean()

In [5]:
class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.projector = MLP(512)

    def forward(self, x):
        z = self.projector(self.encoder(x))
        return F.normalize(z, dim=1)


def nt_xent_loss(z1, z2, temperature=1.0):   # FIXED τ
    N = z1.size(0)
    z = torch.cat([z1, z2], dim=0)

    sim = torch.mm(z, z.t()) / temperature
    mask = torch.eye(2*N, device=z.device).bool()
    sim.masked_fill_(mask, -9e15)

    positives = torch.cat([
        torch.arange(N, 2*N),
        torch.arange(0, N)
    ]).to(z.device)

    return F.cross_entropy(sim, positives)

In [6]:
class LinearProbe(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.fc = nn.Linear(512, 10)

    def forward(self, x):
        with torch.no_grad():
            feat = self.encoder(x)
        return self.fc(feat)


def train_probe(encoder, train_loader, test_loader, epochs=10):
    model = LinearProbe(encoder).to(device)
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for _ in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = criterion(model(x), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

In [7]:
tensor_aug = T.Compose([
    T.RandomResizedCrop(96, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.4,0.4,0.4,0.1),
    T.RandomGrayscale(p=0.2),
])

def jacobian_frobenius(model, x):
    x = x.clone().detach().to(device).requires_grad_(True)
    y = model(x)
    v = torch.randn_like(y)

    Jv = torch.autograd.grad(
        outputs=y,
        inputs=x,
        grad_outputs=v
    )[0]

    return torch.sqrt(Jv.pow(2).sum() / x.size(0))


def compute_metrics(encoder, loader):
    encoder.eval()
    jac_vals, aug_vals, noise_vals = [], [], []

    for i, (x, _) in enumerate(loader):
        if i == 5:
            break

        x = x.to(device)

        jac_vals.append(jacobian_frobenius(encoder, x).item())

        x_aug = torch.stack([tensor_aug(img.cpu()) for img in x]).to(device)

        with torch.no_grad():
            f1 = encoder(x)
            f2 = encoder(x_aug)
            aug_vals.append(torch.norm(f1 - f2, dim=1).mean().item())

        noise = torch.randn_like(x) * 0.1
        with torch.no_grad():
            f2 = encoder(x + noise)
            noise_vals.append(torch.norm(f1 - f2, dim=1).mean().item())

    return np.mean(jac_vals), np.mean(aug_vals), np.mean(noise_vals)

In [9]:


from torch.cuda.amp import autocast, GradScaler

seeds = [0,1,2,3,4,]   
all_results = []

for seed in seeds:

    print("\n====================================")
    print("Running Seed:", seed)
    print("====================================")

    set_seed(seed)
    seed_start = time.time()

    # =============================
    # DATA (TRAIN SPLIT ONLY)
    # =============================

    train_dataset = torchvision.datasets.STL10(
        root="./data",
        split="train",
        download=True,
        transform=ssl_transform
    )

    supervised_train_dataset = torchvision.datasets.STL10(
        root="./data",
        split="train",
        download=False,
        transform=supervised_transform
    )

    test_dataset = torchvision.datasets.STL10(
        root="./data",
        split="test",
        download=False,
        transform=supervised_transform
    )

    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
    supervised_train_loader = DataLoader(supervised_train_dataset, batch_size=128, shuffle=True, num_workers=2)
    supervised_test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

    # =====================================================
    # BYOL
    # =====================================================

    byol_model = BYOL().to(device)
    optimizer = torch.optim.Adam(byol_model.parameters(), lr=3e-4)
    scaler = GradScaler()

    for epoch in range(100):

        epoch_start = time.time()
        byol_model.train()

        for (x1, x2), _ in train_loader:
            x1, x2 = x1.to(device), x2.to(device)

            optimizer.zero_grad()

            with autocast():
                loss = byol_model(x1, x2)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            byol_model.update_target()

        print(f"[BYOL] Epoch {epoch+1}/100 | {time.time()-epoch_start:.2f}s")

    byol_encoder = byol_model.online_encoder

    # =====================================================
    # SimCLR (AMP SAFE LOSS)
    # =====================================================

    simclr_model = SimCLR().to(device)
    optimizer = torch.optim.Adam(simclr_model.parameters(), lr=3e-4)
    scaler = GradScaler()

    for epoch in range(100):

        epoch_start = time.time()
        simclr_model.train()

        for (x1, x2), _ in train_loader:
            x1, x2 = x1.to(device), x2.to(device)

            optimizer.zero_grad()

            with autocast():
                z1 = simclr_model(x1)
                z2 = simclr_model(x2)

                N = z1.size(0)
                z = torch.cat([z1, z2], dim=0).float()
                sim = torch.mm(z, z.t()) / 1.0
                mask = torch.eye(2*N, device=z.device).bool()
                sim.masked_fill_(mask, -1e4)

                positives = torch.cat([
                    torch.arange(N, 2*N),
                    torch.arange(0, N)
                ]).to(z.device)

                loss = F.cross_entropy(sim, positives)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        print(f"[SimCLR] Epoch {epoch+1}/100 | {time.time()-epoch_start:.2f}s")

    simclr_encoder = simclr_model.encoder

    # =====================================================
    # Evaluation
    # =====================================================

    byol_acc = train_probe(byol_encoder, supervised_train_loader, supervised_test_loader)
    simclr_acc = train_probe(simclr_encoder, supervised_train_loader, supervised_test_loader)

    byol_jac, byol_aug, byol_noise = compute_metrics(byol_encoder, supervised_test_loader)
    simclr_jac, simclr_aug, simclr_noise = compute_metrics(simclr_encoder, supervised_test_loader)

    seed_time = (time.time() - seed_start) / 60

    all_results.append({
        "Seed": seed,
        "BYOL_Jacobian": byol_jac,
        "SimCLR_Jacobian": simclr_jac,
        "BYOL_Aug": byol_aug,
        "SimCLR_Aug": simclr_aug,
        "BYOL_Noise": byol_noise,
        "SimCLR_Noise": simclr_noise,
        "BYOL_Acc": byol_acc,
        "SimCLR_Acc": simclr_acc,
        "Seed_Time_Min": seed_time
    })

    pd.DataFrame(all_results).to_csv(
        "stl10_results_progress.csv",
        index=False
    )

    print("Seed finished in", seed_time, "minutes")


Running Seed: 0


100%|██████████| 2.64G/2.64G [04:11<00:00, 10.5MB/s] 
/tmp/ipykernel_55/1203017004.py:50: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_55/1203017004.py:62: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[BYOL] Epoch 1/100 | 28.48s
[BYOL] Epoch 2/100 | 12.31s
[BYOL] Epoch 3/100 | 12.77s
[BYOL] Epoch 4/100 | 12.60s
[BYOL] Epoch 5/100 | 12.42s
[BYOL] Epoch 6/100 | 12.94s
[BYOL] Epoch 7/100 | 12.47s
[BYOL] Epoch 8/100 | 12.49s
[BYOL] Epoch 9/100 | 12.75s
[BYOL] Epoch 10/100 | 12.61s
[BYOL] Epoch 11/100 | 12.35s
[BYOL] Epoch 12/100 | 12.37s
[BYOL] Epoch 13/100 | 12.86s
[BYOL] Epoch 14/100 | 12.59s
[BYOL] Epoch 15/100 | 12.66s
[BYOL] Epoch 16/100 | 12.99s
[BYOL] Epoch 17/100 | 12.72s
[BYOL] Epoch 18/100 | 12.81s
[BYOL] Epoch 19/100 | 13.20s
[BYOL] Epoch 20/100 | 12.67s
[BYOL] Epoch 21/100 | 12.80s
[BYOL] Epoch 22/100 | 12.85s
[BYOL] Epoch 23/100 | 13.02s
[BYOL] Epoch 24/100 | 12.65s
[BYOL] Epoch 25/100 | 12.93s
[BYOL] Epoch 26/100 | 12.60s
[BYOL] Epoch 27/100 | 12.68s
[BYOL] Epoch 28/100 | 12.78s
[BYOL] Epoch 29/100 | 12.79s
[BYOL] Epoch 30/100 | 12.46s
[BYOL] Epoch 31/100 | 12.70s
[BYOL] Epoch 32/100 | 12.72s
[BYOL] Epoch 33/100 | 12.78s
[BYOL] Epoch 34/100 | 12.97s
[BYOL] Epoch 35/100 | 1

/tmp/ipykernel_55/1203017004.py:81: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_55/1203017004.py:93: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[SimCLR] Epoch 1/100 | 13.29s
[SimCLR] Epoch 2/100 | 13.01s
[SimCLR] Epoch 3/100 | 13.12s
[SimCLR] Epoch 4/100 | 12.78s
[SimCLR] Epoch 5/100 | 12.89s
[SimCLR] Epoch 6/100 | 12.92s
[SimCLR] Epoch 7/100 | 13.14s
[SimCLR] Epoch 8/100 | 12.99s
[SimCLR] Epoch 9/100 | 12.82s
[SimCLR] Epoch 10/100 | 12.70s
[SimCLR] Epoch 11/100 | 13.21s
[SimCLR] Epoch 12/100 | 13.03s
[SimCLR] Epoch 13/100 | 12.78s
[SimCLR] Epoch 14/100 | 12.98s
[SimCLR] Epoch 15/100 | 13.26s
[SimCLR] Epoch 16/100 | 13.00s
[SimCLR] Epoch 17/100 | 12.87s
[SimCLR] Epoch 18/100 | 13.17s
[SimCLR] Epoch 19/100 | 13.43s
[SimCLR] Epoch 20/100 | 13.01s
[SimCLR] Epoch 21/100 | 12.81s
[SimCLR] Epoch 22/100 | 12.76s
[SimCLR] Epoch 23/100 | 13.00s
[SimCLR] Epoch 24/100 | 12.95s
[SimCLR] Epoch 25/100 | 13.12s
[SimCLR] Epoch 26/100 | 12.79s
[SimCLR] Epoch 27/100 | 12.79s
[SimCLR] Epoch 28/100 | 13.07s
[SimCLR] Epoch 29/100 | 12.97s
[SimCLR] Epoch 30/100 | 12.79s
[SimCLR] Epoch 31/100 | 13.13s
[SimCLR] Epoch 32/100 | 12.85s
[SimCLR] Epoch 33

In [11]:
results_df

,Seed,BYOL_Jacobian,SimCLR_Jacobian,BYOL_Aug,SimCLR_Aug,BYOL_Noise,SimCLR_Noise,BYOL_Acc,SimCLR_Acc,Seed_Time_Min
0,0,8.060183,23.632816,6.806665,8.843027,7.510628,19.351350,0.464125,0.560000,49.006467
1,1,11.352677,24.481712,6.506293,8.576403,7.206493,21.236852,0.445500,0.546250,42.615301
2,2,9.348303,24.027169,6.830543,8.752125,6.863175,21.907448,0.444250,0.557750,40.688829
3,3,10.143323,22.932073,6.676081,9.272885,9.253421,21.530468,0.468375,0.573375,41.075799
4,4,10.035586,22.983290,6.734712,8.941238,7.923536,19.860251,0.462875,0.563750,42.185518


In [12]:
save_path = "/kaggle/working/stl10_results_progress.csv"
pd.DataFrame(all_results).to_csv(save_path, index=False)
print("Saved to:", save_path)

Saved to: /kaggle/working/stl10_results_progress.csv
